# Tree level 1 — first real tree layer on storage v2

Chunks -> clusters -> `nodes`/`edges` rows -> LLM node summaries. Everything reuses
frozen recipes:
- **Clustering**: the settled pipeline (α=0.5 combined content+path vectors, UMAP-25
  seed 42, HDBSCAN leaf mcs=5, noise chunks attached to nearest clustered chunk in the
  original combined space) — `build_clusters.cluster()` imported unchanged, fed with
  v2 chunk vectors (granite-97m, the experiment model).
- **Summarization**: the chosen two-layer gotcha-aware map-reduce from
  `docs/summarization.md` — per-chunk Haiku condensation cached by content hash
  (same cache dir as the experiments — already-condensed content is free), then the
  `adaptive-gotchas` reduce (Haiku). Prompts + Claude Agent SDK backend imported from
  `summarize_experiments.py` unchanged.
- **References are mechanical**: edges rows, never LLM-written.

Scope: ONE layer, summaries for a 5-node probe (tightest / loosest / largest / median
coherence / seeded-random) — enough to eyeball quality before paying for all nodes.

In [ ]:
import json
import sqlite3
import time
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

import build_clusters as bc

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono-v2.db"
GIT_CACHE = APP_DIR / ".hrono" / "cache" / "git"
MODEL_NAME = "ibm-granite/granite-embedding-97m-multilingual-r2"
ALPHA, BATCH = 0.5, 16

db = sqlite3.connect(DB_PATH)
model = SentenceTransformer(MODEL_NAME)

# chunk bodies from cache byte spans; embed whatever isn't in the embeddings table yet
chunk_rows = db.execute(
    "SELECT id, instance, path, byte_start, byte_end, hash, chunk_no, uri FROM chunks"
).fetchall()
body_by_hash = {}
for _, inst, path, s, e, h, _, _ in chunk_rows:
    if h not in body_by_hash:
        raw = (GIT_CACHE / inst / path).read_bytes()
        body_by_hash[h] = raw[s:e].decode("utf-8", "replace")

done = {r[0] for r in db.execute("SELECT hash FROM embeddings WHERE model = ?", (MODEL_NAME,))}
todo = [h for h in body_by_hash if h not in done]
print(f"{len(body_by_hash)} unique chunk bodies, {len(todo)} to embed")
for i in range(0, len(todo), BATCH):
    batch = todo[i:i + BATCH]
    vecs = model.encode([body_by_hash[h] for h in batch], batch_size=BATCH,
                        normalize_embeddings=True, show_progress_bar=False)
    with db:
        db.executemany("INSERT INTO embeddings (hash, model, dims, vector) VALUES (?,?,?,?)",
                       [(h, MODEL_NAME, vecs.shape[1], np.asarray(v, np.float32).tobytes())
                        for h, v in zip(batch, vecs)])

# one row per unique hash (dedup like the v1 pipeline); representative path per hash
rows = db.execute("""
    SELECT c.hash, c.path, e.vector FROM chunks c
    JOIN embeddings e ON e.hash = c.hash AND e.model = ?
    GROUP BY c.hash""", (MODEL_NAME,)).fetchall()
hashes = [r[0] for r in rows]
chunk_paths = [r[1] for r in rows]
content_vecs = np.frombuffer(b"".join(r[2] for r in rows), np.float32).reshape(len(rows), -1)

distinct_paths = sorted(set(chunk_paths))
pv = dict(zip(distinct_paths, model.encode(distinct_paths, batch_size=BATCH,
                                           normalize_embeddings=True, show_progress_bar=False)))
path_vecs = np.stack([pv[p] for p in chunk_paths])
combined = np.hstack([np.sqrt(1 - ALPHA) * content_vecs, np.sqrt(ALPHA) * path_vecs])
print(f"{len(hashes)} unique-hash chunks, combined dims {combined.shape[1]}")

## Cluster and materialize the layer: `nodes` + `edges`

Schema kept minimal for the first layer: a node is a cluster at a level; an edge binds a
node to its children (chunks at level 1; nodes at level 2+, hence child_kind). Summary
lands on the node row once generated — with the model and cost, because provenance of
generated text is data too.

In [ ]:
labels, assigned, assignment_sim = bc.cluster(combined)

# summaries survive rebuilds: keyed by the node's member-set signature (sorted member
# hashes), the germ of the future diff layer — unchanged membership = reusable summary
try:
    saved = {r[0]: r[1:] for r in db.execute(
        "SELECT sig, summary, summary_model, summary_cost_usd, summary_prompt_id,"
        " summary_created_at FROM nodes WHERE summary IS NOT NULL")}
except sqlite3.OperationalError:
    try:  # pre-provenance schema: carry what exists
        saved = {r[0]: r[1:] + (None, None) for r in db.execute(
            "SELECT sig, summary, summary_model, summary_cost_usd FROM nodes"
            " WHERE summary IS NOT NULL")}
    except sqlite3.OperationalError:
        saved = {}
print(f"{len(saved)} existing summaries carried over by signature")

db.executescript("""
DROP TABLE IF EXISTS edges;
DROP TABLE IF EXISTS nodes;
CREATE TABLE nodes (
    id        INTEGER PRIMARY KEY,
    level     INTEGER NOT NULL,          -- 1 = clusters of chunks
    size      INTEGER NOT NULL,          -- member count
    coherence REAL    NOT NULL,          -- mean pairwise cosine of member content vectors
    sig       TEXT    NOT NULL,          -- sha256 of sorted member hashes: summary reuse key
    summary   TEXT,                      -- NULL until generated
    summary_model TEXT,
    summary_cost_usd REAL,
    summary_prompt_id INTEGER REFERENCES prompts(id),
    summary_created_at TEXT              -- ISO-8601 UTC
);
CREATE TABLE edges (
    parent_id  INTEGER NOT NULL REFERENCES nodes(id),
    child_kind TEXT    NOT NULL,         -- 'chunk' (level 1) | 'node' (level 2+)
    child_id   INTEGER NOT NULL,         -- chunks.id or nodes.id
    assigned   INTEGER NOT NULL,         -- 1 = noise chunk attached by proximity, not density
    UNIQUE (parent_id, child_kind, child_id)
);
""")

chunk_ids_by_hash = {}
for cid, inst, path, s, e, h, chunk_no, uri in chunk_rows:
    chunk_ids_by_hash.setdefault(h, []).append(cid)

with db:
    for label in sorted(set(labels)):
        idx = [i for i, l in enumerate(labels) if l == label]
        coherence = bc.mean_pairwise_cosine(content_vecs[idx])
        members = [(cid, bool(assigned[i])) for i in idx for cid in chunk_ids_by_hash[hashes[i]]]
        from hashlib import sha256 as _sha
        sig = _sha("".join(sorted(hashes[i] for i in idx)).encode()).hexdigest()
        summary, s_model, s_cost, s_prompt, s_created = saved.get(
            sig, (None, None, None, None, None))
        cur = db.execute(
            "INSERT INTO nodes (level, size, coherence, sig, summary, summary_model,"
            " summary_cost_usd, summary_prompt_id, summary_created_at)"
            " VALUES (1, ?, ?, ?, ?, ?, ?, ?, ?)",
            (len(members), coherence, sig, summary, s_model, s_cost, s_prompt, s_created))
        db.executemany("INSERT INTO edges (parent_id, child_kind, child_id, assigned)"
                       " VALUES (?, 'chunk', ?, ?)",
                       [(cur.lastrowid, cid, int(a)) for cid, a in members])

import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 120)
nodes_df = pd.read_sql_query("""
    SELECT n.id, n.level, n.size, n.coherence,
           (SELECT GROUP_CONCAT(DISTINCT substr(c.path, 1, instr(c.path || '/', '/') +
                   instr(substr(c.path || '/', instr(c.path || '/', '/') + 1), '/') - 1))
            FROM edges e JOIN chunks c ON c.id = e.child_id
            WHERE e.parent_id = n.id LIMIT 3) AS repos
    FROM nodes n ORDER BY n.coherence DESC""", db)
print(f"{len(nodes_df)} level-1 nodes, sizes {sorted(nodes_df['size'].tolist(), reverse=True)}")
display(nodes_df)

## Summarize the 5-node probe

Recipe verbatim from the experiments: map = `condense_chunk` (Haiku, GOTCHA lines,
cached by content hash in the shared map-cache), reduce = `adaptive-gotchas` prompt
(Haiku) over the condensations. Chunk labels are real paths so the summary can name
files. Probe: tightest, loosest, largest, median-coherence, seeded-random.

In [ ]:
import random
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import summarize_experiments as se

from datetime import datetime, timezone

# prompts are data: every generated text references the exact prompt that produced it.
# (name, text) unique — editing a prompt creates a NEW row; history stays truthful.
db.executescript("""
CREATE TABLE IF NOT EXISTS prompts (
    id   INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    text TEXT NOT NULL,
    UNIQUE (name, text)
);
CREATE TABLE IF NOT EXISTS condensations (
    hash       TEXT PRIMARY KEY,   -- chunk content hash
    model      TEXT NOT NULL,
    text       TEXT NOT NULL,
    prompt_id  INTEGER REFERENCES prompts(id),
    created_at TEXT                -- ISO-8601 UTC
);
""")
for col, ddl in (("prompt_id", "INTEGER REFERENCES prompts(id)"), ("created_at", "TEXT")):
    if col not in {r[1] for r in db.execute("PRAGMA table_info(condensations)")}:
        db.execute(f"ALTER TABLE condensations ADD COLUMN {col} {ddl}")


def prompt_id(name, text):
    with db:
        db.execute("INSERT OR IGNORE INTO prompts (name, text) VALUES (?,?)", (name, text))
    return db.execute("SELECT id FROM prompts WHERE name = ? AND text = ?",
                      (name, text)).fetchone()[0]


def utcnow():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


# verbatim copy of se.condense_chunk's inline template (that file is experiment
# history — not edited; keep this in sync manually if the recipe ever changes)
MAP_PROMPT = (
    "Condense this content chunk to its essential information. Let the length "
    "follow the density of the content: boilerplate, generated, or repetitive "
    "content collapses to one short line saying so; dense logic keeps whatever "
    "it needs to stay faithful. Copy concrete identifiers, names, values, and "
    "constraints VERBATIM from the content - never infer or add ones that are "
    "not literally there. If the content holds anything non-obvious a newcomer "
    "could be hurt by (behavior differing from what names suggest, silent "
    "no-ops, partial or non-transactional operations, asymmetries, workarounds, "
    "fragile special cases), add a line starting with 'GOTCHA:' for each - only "
    "if literally present. Output only the condensed content."
)
MAP_PROMPT_ID = prompt_id("map-condense", MAP_PROMPT)
REDUCE_PROMPT_ID = prompt_id("adaptive-gotchas", se.PROMPTS["adaptive-gotchas"])

# backfill provenance on rows created before these columns existed
with db:
    db.execute("UPDATE condensations SET prompt_id = ?, created_at = COALESCE(created_at, ?)"
               " WHERE prompt_id IS NULL", (MAP_PROMPT_ID, utcnow()))
    db.execute("UPDATE nodes SET summary_prompt_id = ?,"
               " summary_created_at = COALESCE(summary_created_at, ?)"
               " WHERE summary IS NOT NULL AND summary_prompt_id IS NULL",
               (REDUCE_PROMPT_ID, utcnow()))


def condense_all(chunks):
    """DB-first map stage. SQLite is main-thread-only here: lookups and inserts stay on
    this thread; only the Claude calls (misses) go to the pool."""
    out, missing = {}, []
    for c in chunks:
        row = db.execute("SELECT text FROM condensations WHERE hash = ?", (c.hash,)).fetchone()
        if row:
            out[c.label] = row[0]
        else:
            missing.append(c)
    if missing:
        with ThreadPoolExecutor(max_workers=se.PARALLEL_CALLS) as pool:
            results = list(pool.map(se.condense_chunk, missing))  # claude or se's file cache
        out.update({c.label: t for c, t in zip(missing, results)})
        with db:
            db.executemany(
                "INSERT OR IGNORE INTO condensations (hash, model, text, prompt_id, created_at)"
                " VALUES (?,?,?,?,?)",
                [(c.hash, se.MAP_MODEL, t, MAP_PROMPT_ID, utcnow())
                 for c, t in zip(missing, results)])
    print(f"  map: {len(chunks) - len(missing)} from DB, {len(missing)} new calls")
    return out


def node_chunks(node_id):
    return db.execute("""
        SELECT c.id, c.path, c.chunk_no, c.hash, c.instance, c.byte_start, c.byte_end
        FROM edges e JOIN chunks c ON c.id = e.child_id
        WHERE e.parent_id = ? ORDER BY c.path, c.chunk_no""", (node_id,)).fetchall()


@dataclass
class C:  # duck-typed stand-in for summarize_experiments.Chunk
    label: str
    hash: str
    text: str


def chunks_for_summary(node_id):
    out = []
    for cid, path, chunk_no, h, inst, s, e in node_chunks(node_id):
        raw = (GIT_CACHE / inst / path).read_bytes()
        out.append(C(label=f"{path}#chunk-{chunk_no}", hash=h,
                     text=raw[s:e].decode("utf-8", "replace")))
    return out


ordered = nodes_df.sort_values("coherence")
rng = random.Random(42)
probe = sorted({int(ordered.iloc[-1]["id"]),                       # tightest
                int(ordered.iloc[0]["id"]),                        # loosest
                int(nodes_df.sort_values("size").iloc[-1]["id"]),  # largest
                int(ordered.iloc[len(ordered) // 2]["id"]),        # median coherence
                int(rng.choice(nodes_df["id"].tolist()))})         # seeded random
print(f"probe nodes: {probe}")

for node_id in probe:
    if db.execute("SELECT summary FROM nodes WHERE id = ?", (node_id,)).fetchone()[0]:
        print(f"node {node_id}: summary exists, skipping")
        continue
    chunks = chunks_for_summary(node_id)
    print(f"node {node_id}: {len(chunks)} chunks — map stage...")
    condensed = condense_all(chunks)
    content = se.render_condensed(chunks, condensed)
    t0 = time.monotonic()
    # call_claude uses asyncio.run, illegal on Jupyter's event-loop thread -> worker thread
    with ThreadPoolExecutor(max_workers=1) as ex:
        result = ex.submit(se.call_claude,
                           se.instructions("adaptive-gotchas") + "\n\n" + content,
                           "haiku").result()
    with db:
        db.execute("UPDATE nodes SET summary = ?, summary_model = ?, summary_cost_usd = ?,"
                   " summary_prompt_id = ?, summary_created_at = ? WHERE id = ?",
                   (result["text"], "haiku:adaptive-gotchas-condensed", result["cost_usd"],
                    REDUCE_PROMPT_ID, utcnow(), node_id))
    print(f"  reduce done in {time.monotonic() - t0:.0f}s, ${result['cost_usd']:.3f}")

## The summaries — eyeball review

Each probe node: its stats, member listing (mechanical, from edges), then the generated
summary. Judge: does it state what the group IS, use real names, compress repetition,
and surface believable gotchas?

In [ ]:
for node_id in probe:
    size, coherence, summary, cost = db.execute(
        "SELECT size, coherence, summary, summary_cost_usd FROM nodes WHERE id = ?",
        (node_id,)).fetchone()
    members = node_chunks(node_id)
    paths = sorted({m[1] for m in members})
    listing = "\n".join(f"- `{p}`" for p in paths[:15])
    if len(paths) > 15:
        listing += f"\n- … {len(paths) - 15} more files"
    display(Markdown(
        f"---\n## node {node_id} — {size} chunks, coherence {coherence:.3f}, "
        f"cost ${cost or 0:.3f}\n**Members ({len(paths)} files):**\n{listing}\n\n"
        f"**Summary:**\n\n{summary or '(none)'}"))

total = db.execute("SELECT COUNT(*), SUM(summary_cost_usd) FROM nodes WHERE summary IS NOT NULL").fetchone()
print(f"\nsummarized {total[0]} nodes, total ${total[1] or 0:.3f}")
db.close()

## Observations

(To fill after eyeball review. Open items carried in: Sonnet-for-messy-clusters heuristic
untested; level-2 composition next; the diff layer — only changed-membership nodes
re-summarize — is the update mechanism's job once the layer stabilizes.)